# Analyze HelpSteer2 All-Method Results

This notebook creates compact comparison tables and bar charts for the completed HelpSteer2 all-method merge evaluation. It compares method quality, method cost, and utility-vs-cost behavior inside the fixed Rewarded-Soups-style interpolation family.

All scores are proxy scores. They are not HelpSteer2 human labels, not external reward-model scores, and do not establish global Pareto-front improvement.


## 1. Clone or update repository


In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")


## 2. Install analysis dependencies


In [ ]:
!pip install -q "pandas==2.2.2" "numpy<2.1" matplotlib tabulate


## 3. Load completed evaluation outputs


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

summary_path = Path("results/helpsteer2_all_method_result_summary.csv")
scores_path = Path("results/helpsteer2_all_method_scores.csv")
generations_path = Path("results/helpsteer2_all_method_generations.csv")
costs_path = Path("results/helpsteer2_method_costs.csv")
coefficients_path = Path("results/helpsteer2_all_method_coefficients.csv")

required_paths = [summary_path, scores_path, generations_path, costs_path, coefficients_path]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError("Run Notebook 17 first. Missing: " + ", ".join(str(path) for path in missing))

summary = pd.read_csv(summary_path)
scores = pd.read_csv(scores_path)
generations = pd.read_csv(generations_path)
costs = pd.read_csv(costs_path)
coefficients = pd.read_csv(coefficients_path)

print(f"Summary rows: {len(summary)}")
print(f"Score rows: {len(scores)}")
print(f"Generation rows: {len(generations)}")
print(f"Cost rows: {len(costs)}")
display(summary.head())


## 4. Build method-level tables


In [ ]:
for column in [
    "mean_utility",
    "improvement_over_direct_preference",
    "improvement_over_uniform",
    "l1_distance_to_p",
    "l2_distance_to_p",
    "runtime_seconds",
    "peak_memory_mb",
    "solver_iterations",
]:
    if column in summary.columns:
        summary[column] = pd.to_numeric(summary[column], errors="coerce")
    if column in costs.columns:
        costs[column] = pd.to_numeric(costs[column], errors="coerce")

method_family = summary.groupby("method")["method_family"].first().reset_index()

performance_summary = (
    summary.groupby("method", as_index=False)
    .agg(
        mean_utility=("mean_utility", "mean"),
        std_utility=("mean_utility", "std"),
        best_utility=("mean_utility", "max"),
        mean_improvement_over_direct=("improvement_over_direct_preference", "mean"),
        mean_improvement_over_uniform=("improvement_over_uniform", "mean"),
        mean_l1_distance_to_p=("l1_distance_to_p", "mean"),
        mean_l2_distance_to_p=("l2_distance_to_p", "mean"),
        num_runs=("mean_utility", "size"),
    )
    .sort_values("mean_utility", ascending=False)
)

cost_summary = (
    costs.groupby("method", as_index=False)
    .agg(
        mean_runtime_seconds=("runtime_seconds", "mean"),
        total_runtime_seconds=("runtime_seconds", "sum"),
        mean_peak_memory_mb=("peak_memory_mb", "mean"),
        mean_solver_iterations=("solver_iterations", "mean"),
        num_runs=("runtime_seconds", "size"),
    )
    .sort_values("mean_runtime_seconds")
)

comparison_table = (
    performance_summary.merge(method_family, on="method", how="left")
    .merge(
        cost_summary[[
            "method",
            "mean_runtime_seconds",
            "mean_peak_memory_mb",
            "mean_solver_iterations",
        ]],
        on="method",
        how="left",
    )
)
comparison_table = comparison_table[[
    "method",
    "method_family",
    "num_runs",
    "mean_utility",
    "std_utility",
    "best_utility",
    "mean_improvement_over_direct",
    "mean_improvement_over_uniform",
    "mean_l1_distance_to_p",
    "mean_l2_distance_to_p",
    "mean_runtime_seconds",
    "mean_peak_memory_mb",
    "mean_solver_iterations",
]].sort_values("mean_utility", ascending=False)

display(comparison_table)
display(cost_summary)
display(performance_summary)


## 5. Save compact CSV tables


In [ ]:
comparison_path = Path("results/helpsteer2_method_comparison_table.csv")
cost_summary_path = Path("results/helpsteer2_method_cost_summary.csv")
performance_summary_path = Path("results/helpsteer2_method_performance_summary.csv")

comparison_table.to_csv(comparison_path, index=False)
cost_summary.to_csv(cost_summary_path, index=False)
performance_summary.to_csv(performance_summary_path, index=False)

print(f"Saved {comparison_path}")
print(f"Saved {cost_summary_path}")
print(f"Saved {performance_summary_path}")


## 6. Best row per preference


In [ ]:
best_row_per_preference = (
    summary.sort_values("mean_utility", ascending=False)
    .groupby("preference_name", as_index=False)
    .first()
)

best_columns = [
    "preference_name",
    "method",
    "hyperparameter_id",
    "mean_utility",
    "l1_distance_to_p",
    "l2_distance_to_p",
    "runtime_seconds",
]
display(best_row_per_preference[best_columns])


## 7. Create bar charts


In [ ]:
plots_dir = Path("results/plots")
plots_dir.mkdir(parents=True, exist_ok=True)

utility_plot_path = plots_dir / "helpsteer2_method_mean_utility_bar.png"
runtime_plot_path = plots_dir / "helpsteer2_method_runtime_bar.png"
utility_cost_plot_path = plots_dir / "helpsteer2_method_utility_vs_cost_bar.png"

plot_table = comparison_table.copy()
method_order = ["direct_preference", "uniform", "M1", "M2", "C1", "C2", "P1", "P2"]
plot_table["method"] = pd.Categorical(plot_table["method"], categories=method_order, ordered=True)
plot_table = plot_table.sort_values("method")

def save_bar(path, x, y, title, ylabel, color):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x, y, color=color)
    ax.set_title(title)
    ax.set_xlabel("Method")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.show()

save_bar(
    utility_plot_path,
    plot_table["method"].astype(str),
    plot_table["mean_utility"],
    "Mean Proxy Utility by Method",
    "Mean proxy utility",
    "#4C78A8",
)

save_bar(
    runtime_plot_path,
    plot_table["method"].astype(str),
    plot_table["mean_runtime_seconds"],
    "Mean Coefficient Runtime by Method",
    "Mean runtime (seconds)",
    "#F58518",
)

normalized = plot_table.copy()
normalized["mean_utility_norm"] = normalized["mean_utility"] / normalized["mean_utility"].max()
normalized["mean_runtime_norm"] = normalized["mean_runtime_seconds"] / normalized["mean_runtime_seconds"].max()

x = np.arange(len(normalized))
width = 0.38
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, normalized["mean_utility_norm"], width, label="Mean utility (normalized)", color="#4C78A8")
ax.bar(x + width / 2, normalized["mean_runtime_norm"], width, label="Mean runtime (normalized)", color="#F58518")
ax.set_title("Mean Utility vs Coefficient Runtime by Method")
ax.set_xlabel("Method")
ax.set_ylabel("Normalized value")
ax.set_xticks(x)
ax.set_xticklabels(normalized["method"].astype(str), rotation=30, ha="right")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(utility_cost_plot_path, dpi=180)
plt.show()

print(f"Saved {utility_plot_path}")
print(f"Saved {runtime_plot_path}")
print(f"Saved {utility_cost_plot_path}")


## 8. Save compact Markdown analysis report


In [ ]:
report_path = Path("results/helpsteer2_all_method_analysis_report.md")

top_method = comparison_table.iloc[0]
cheapest_non_baseline = cost_summary[~cost_summary["method"].isin(["direct_preference", "uniform"])].iloc[0]

lines = [
    "# HelpSteer2 All-Method Analysis Report",
    "",
    "This compact report summarizes the completed HelpSteer2 all-method merge evaluation with table-focused comparisons and chart outputs. The comparison is inside the fixed Rewarded-Soups-style interpolation family.",
    "",
    "All quality values are proxy scores. They are not HelpSteer2 human labels, not external reward-model scores, and do not establish global Pareto-front improvement.",
    "",
    "## Best Row Per Preference",
    "",
    best_row_per_preference[best_columns].to_markdown(index=False),
    "",
    "## Method Comparison Table",
    "",
    comparison_table.to_markdown(index=False),
    "",
    "## Computational Cost Summary",
    "",
    cost_summary.to_markdown(index=False),
    "",
    "## Performance Summary",
    "",
    performance_summary.to_markdown(index=False),
    "",
    "## Chart Files",
    "",
    "- `results/plots/helpsteer2_method_mean_utility_bar.png`: mean proxy utility by method.",
    "- `results/plots/helpsteer2_method_runtime_bar.png`: mean coefficient-computation runtime by method.",
    "- `results/plots/helpsteer2_method_utility_vs_cost_bar.png`: normalized mean utility and normalized mean runtime by method.",
    "",
    "## Interpretation",
    "",
    f"- `{top_method['method']}` has the highest average proxy utility in this method-level view.",
    "- The best individual settings per preference are split between `C2` and `M2`, so the promising method family depends on the preference vector.",
    f"- `{cheapest_non_baseline['method']}` is the cheapest non-baseline method in the coefficient-computation cost table.",
    "- Stronger proxy utility can come with larger movement away from the original preference vector `p`; this should be treated as a utility-vs-preference-faithfulness tradeoff.",
    "- Another targeted hyperparameter round is most useful around the currently promising `C2` and `M2` settings.",
    "",
    "## Limitations",
    "",
    "- Proxy scores are deterministic heuristics and may not reflect real HelpSteer2 human preferences.",
    "- GPT-2 is a small prototype model and can produce low-quality generations.",
    "- The fixed prompt set improves reproducibility but is still limited.",
    "- Runtime numbers measure coefficient computation, not the full generation cost.",
    "- The results compare choices inside one fixed interpolation family and do not show global Pareto-front improvement.",
]

report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Saved {report_path}")


## 9. Preview report and git status


In [ ]:
!cat results/helpsteer2_all_method_analysis_report.md
!git status
